# Customer Churn & Retention Analysis

## Project Overview

This notebook analyzes the cleaned Customer Churn & Retention Analysis datasets using Python.

The analysis focuses on:
- Data loading and validation
- Customer churn and retention KPIs
- Churn by customer segment, contract, subscription, and payment method
- Customer engagement and churn behavior
- Tenure and demographic analysis
- Churn reasons and churn trends
- Revenue at risk
- High-risk customer identification
- Business insights and recommendations

The notebook uses the supplied cleaned datasets and does not modify the original files.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries loaded successfully.")


## 1. Load the Cleaned Datasets

The project uses three cleaned datasets:
- `churn_cleaned.csv`
- `customer_activity_cleaned.csv`
- `subscriptions_cleaned.csv`


In [ ]:
# Update DATA_PATH if your CSV files are stored in another folder.
DATA_PATH = "."

churn = pd.read_csv(f"{DATA_PATH}/churn_cleaned.csv")
activity = pd.read_csv(f"{DATA_PATH}/customer_activity_cleaned.csv")
subscriptions = pd.read_csv(f"{DATA_PATH}/subscriptions_cleaned.csv")

print("Churn:", churn.shape)
print("Customer Activity:", activity.shape)
print("Subscriptions:", subscriptions.shape)


In [ ]:
print("Churn columns:")
print(churn.columns.tolist())

print("\nCustomer Activity columns:")
print(activity.columns.tolist())

print("\nSubscriptions columns:")
print(subscriptions.columns.tolist())


## 2. Data Quality Checks

In [ ]:
# Preview each dataset
display(churn.head())
display(activity.head())
display(subscriptions.head())


In [ ]:
# Missing values
quality_summary = pd.DataFrame({
    "churn_missing": churn.isna().sum(),
    "activity_missing": activity.isna().sum(),
    "subscription_missing": subscriptions.isna().sum()
})

display(quality_summary)


In [ ]:
# Duplicate checks
print("Duplicate rows:")
print("Churn:", churn.duplicated().sum())
print("Activity:", activity.duplicated().sum())
print("Subscriptions:", subscriptions.duplicated().sum())

print("\nDuplicate customer IDs:")
print("Churn:", churn["customer_id"].duplicated().sum())
print("Activity:", activity["customer_id"].duplicated().sum())
print("Subscriptions:", subscriptions["customer_id"].duplicated().sum())


In [ ]:
# Basic numeric validation
numeric_columns = [
    "age",
    "monthly_charge",
    "login_count",
    "support_tickets",
    "avg_session_minutes",
    "monthly_usage",
    "tenure_months"
]

available_numeric = [c for c in numeric_columns if c in activity.columns]
display(activity[available_numeric].describe().T)


## 3. Prepare the Analysis Dataset

In [ ]:
# Merge churn status with customer activity and subscription information.
analysis = (
    activity
    .merge(churn, on="customer_id", how="left", suffixes=("", "_churn"))
    .merge(
        subscriptions,
        on="customer_id",
        how="left",
        suffixes=("", "_subscription")
    )
)

analysis["customer_status"] = np.where(
    analysis["churn_flag"] == 1,
    "Churned",
    "Retained"
)

print("Analysis dataset shape:", analysis.shape)
display(analysis.head())


## 4. Overall Churn & Retention KPIs

In [ ]:
total_customers = analysis["customer_id"].nunique()
churned_customers = analysis.loc[analysis["churn_flag"] == 1, "customer_id"].nunique()
retained_customers = total_customers - churned_customers

churn_rate = churned_customers / total_customers * 100
retention_rate = retained_customers / total_customers * 100

kpis = pd.DataFrame({
    "KPI": [
        "Total Customers",
        "Churned Customers",
        "Retained Customers",
        "Churn Rate (%)",
        "Retention Rate (%)"
    ],
    "Value": [
        total_customers,
        churned_customers,
        retained_customers,
        churn_rate,
        retention_rate
    ]
})

display(kpis)


In [ ]:
status_summary = (
    analysis.groupby("customer_status")
    .agg(customers=("customer_id", "nunique"))
    .reset_index()
)

status_summary["percentage"] = (
    status_summary["customers"] / status_summary["customers"].sum() * 100
)

display(status_summary)


## 5. Churn by Customer Segment

In [ ]:
segment_analysis = (
    analysis.groupby("customer_segment")
    .agg(
        total_customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

segment_analysis["churn_rate_pct"] = (
    segment_analysis["churned_customers"] /
    segment_analysis["total_customers"] * 100
)

segment_analysis = segment_analysis.sort_values(
    "churn_rate_pct", ascending=False
)

display(segment_analysis)


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(
    segment_analysis["customer_segment"],
    segment_analysis["churn_rate_pct"]
)
plt.title("Churn Rate by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 6. Churn by Contract Type

In [ ]:
contract_analysis = (
    analysis.groupby("contract_type")
    .agg(
        total_customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

contract_analysis["churn_rate_pct"] = (
    contract_analysis["churned_customers"] /
    contract_analysis["total_customers"] * 100
)

contract_analysis = contract_analysis.sort_values(
    "churn_rate_pct", ascending=False
)

display(contract_analysis)


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(
    contract_analysis["contract_type"],
    contract_analysis["churn_rate_pct"]
)
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 7. Churn by Subscription Type

In [ ]:
subscription_analysis = (
    analysis.groupby("subscription_type")
    .agg(
        total_customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

subscription_analysis["churn_rate_pct"] = (
    subscription_analysis["churned_customers"] /
    subscription_analysis["total_customers"] * 100
)

subscription_analysis = subscription_analysis.sort_values(
    "churn_rate_pct", ascending=False
)

display(subscription_analysis)


## 8. Churn by Payment Method

In [ ]:
payment_analysis = (
    analysis.groupby("payment_method")
    .agg(
        total_customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

payment_analysis["churn_rate_pct"] = (
    payment_analysis["churned_customers"] /
    payment_analysis["total_customers"] * 100
)

payment_analysis = payment_analysis.sort_values(
    "churn_rate_pct", ascending=False
)

display(payment_analysis)


## 9. Customer Engagement and Churn

In [ ]:
engagement_summary = (
    analysis.groupby("customer_status")
    .agg(
        customers=("customer_id", "nunique"),
        avg_login_count=("login_count", "mean"),
        avg_support_tickets=("support_tickets", "mean"),
        avg_session_minutes=("avg_session_minutes", "mean"),
        avg_monthly_usage=("monthly_usage", "mean"),
        avg_monthly_charge=("monthly_charge", "mean"),
        avg_tenure_months=("tenure_months", "mean")
    )
    .reset_index()
)

display(engagement_summary)


In [ ]:
# Compare selected engagement metrics
metrics = [
    "login_count",
    "support_tickets",
    "avg_session_minutes",
    "monthly_usage"
]

for metric in metrics:
    comparison = analysis.groupby("customer_status")[metric].mean()
    print(f"\n{metric}")
    print(comparison)


## 10. Login Activity and Churn

In [ ]:
analysis["login_band"] = pd.cut(
    analysis["login_count"],
    bins=[-np.inf, 5, 10, 15, np.inf],
    labels=["0-5", "6-10", "11-15", "16+"]
)

login_analysis = (
    analysis.groupby("login_band", observed=True)
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

login_analysis["churn_rate_pct"] = (
    login_analysis["churned_customers"] /
    login_analysis["customers"] * 100
)

display(login_analysis)


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(
    login_analysis["login_band"].astype(str),
    login_analysis["churn_rate_pct"]
)
plt.title("Churn Rate by Login Activity")
plt.xlabel("Login Activity Band")
plt.ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()


## 11. Support Tickets and Churn

In [ ]:
analysis["ticket_band"] = pd.cut(
    analysis["support_tickets"],
    bins=[-np.inf, 0, 2, 5, np.inf],
    labels=["0", "1-2", "3-5", "6+"],
    right=True
)

ticket_analysis = (
    analysis.groupby("ticket_band", observed=True)
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

ticket_analysis["churn_rate_pct"] = (
    ticket_analysis["churned_customers"] /
    ticket_analysis["customers"] * 100
)

display(ticket_analysis)


## 12. Monthly Usage and Churn

In [ ]:
analysis["usage_band"] = pd.cut(
    analysis["monthly_usage"],
    bins=[-np.inf, 20, 40, 60, np.inf],
    labels=["<20", "20-39.9", "40-59.9", "60+"],
    right=False
)

usage_analysis = (
    analysis.groupby("usage_band", observed=True)
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

usage_analysis["churn_rate_pct"] = (
    usage_analysis["churned_customers"] /
    usage_analysis["customers"] * 100
)

display(usage_analysis)


## 13. Tenure Analysis

In [ ]:
analysis["tenure_band"] = pd.cut(
    analysis["tenure_months"],
    bins=[-np.inf, 12, 24, 36, 48, np.inf],
    labels=["<12 months", "12-23 months", "24-35 months", "36-47 months", "48+ months"],
    right=False
)

tenure_analysis = (
    analysis.groupby("tenure_band", observed=True)
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

tenure_analysis["churn_rate_pct"] = (
    tenure_analysis["churned_customers"] /
    tenure_analysis["customers"] * 100
)

display(tenure_analysis)


In [ ]:
tenure_status = (
    analysis.groupby("customer_status")["tenure_months"]
    .agg(["count", "mean", "min", "max"])
    .reset_index()
)

display(tenure_status)


## 14. Demographic Analysis

In [ ]:
gender_analysis = (
    analysis.groupby("gender")
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

gender_analysis["churn_rate_pct"] = (
    gender_analysis["churned_customers"] /
    gender_analysis["customers"] * 100
)

display(gender_analysis)


In [ ]:
analysis["age_band"] = pd.cut(
    analysis["age"],
    bins=[-np.inf, 25, 35, 45, 55, 65, np.inf],
    labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"],
    right=False
)

age_analysis = (
    analysis.groupby("age_band", observed=True)
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

age_analysis["churn_rate_pct"] = (
    age_analysis["churned_customers"] /
    age_analysis["customers"] * 100
)

display(age_analysis)


## 15. Churn by City

In [ ]:
city_analysis = (
    analysis.groupby("city")
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

city_analysis["churn_rate_pct"] = (
    city_analysis["churned_customers"] /
    city_analysis["customers"] * 100
)

# Focus on cities with at least 50 customers for a more stable comparison.
city_analysis_50 = (
    city_analysis[city_analysis["customers"] >= 50]
    .sort_values("churn_rate_pct", ascending=False)
)

display(city_analysis_50.head(15))


## 16. Churn Reasons

In [ ]:
churn_reasons = (
    analysis.loc[analysis["churn_flag"] == 1]
    .groupby("churn_reason")
    .size()
    .reset_index(name="churned_customers")
    .sort_values("churned_customers", ascending=False)
)

churn_reasons["share_of_churn_pct"] = (
    churn_reasons["churned_customers"] /
    churn_reasons["churned_customers"].sum() * 100
)

display(churn_reasons)


In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(
    churn_reasons["churn_reason"].astype(str),
    churn_reasons["churned_customers"]
)
plt.title("Churned Customers by Churn Reason")
plt.xlabel("Churned Customers")
plt.ylabel("Churn Reason")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 17. Churn Trend Over Time

In [ ]:
analysis["churn_date"] = pd.to_datetime(
    analysis["churn_date"],
    errors="coerce"
)

monthly_churn = (
    analysis.loc[analysis["churn_flag"] == 1]
    .dropna(subset=["churn_date"])
    .assign(churn_month=lambda df: df["churn_date"].dt.to_period("M").astype(str))
    .groupby("churn_month")
    .size()
    .reset_index(name="churned_customers")
)

display(monthly_churn)


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(
    monthly_churn["churn_month"],
    monthly_churn["churned_customers"],
    marker="o"
)
plt.title("Monthly Churn Trend")
plt.xlabel("Month")
plt.ylabel("Churned Customers")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 18. Revenue at Risk

In [ ]:
churned = analysis[analysis["churn_flag"] == 1].copy()

monthly_revenue_at_risk = churned["monthly_charge"].sum()
avg_churned_charge = churned["monthly_charge"].mean()

retained = analysis[analysis["churn_flag"] == 0].copy()
avg_retained_charge = retained["monthly_charge"].mean()

revenue_risk_summary = pd.DataFrame({
    "Metric": [
        "Churned Customers",
        "Monthly Revenue at Risk",
        "Average Monthly Charge - Churned",
        "Average Monthly Charge - Retained"
    ],
    "Value": [
        len(churned),
        monthly_revenue_at_risk,
        avg_churned_charge,
        avg_retained_charge
    ]
})

display(revenue_risk_summary)


In [ ]:
revenue_risk_by_segment = (
    churned.groupby("customer_segment")
    .agg(
        churned_customers=("customer_id", "nunique"),
        monthly_revenue_at_risk=("monthly_charge", "sum"),
        avg_monthly_charge=("monthly_charge", "mean")
    )
    .reset_index()
    .sort_values("monthly_revenue_at_risk", ascending=False)
)

display(revenue_risk_by_segment)


## 19. High-Value Churned Customers

In [ ]:
high_value_churned = (
    churned[
        [
            "customer_id",
            "customer_segment",
            "contract_type",
            "monthly_charge",
            "login_count",
            "support_tickets",
            "monthly_usage",
            "tenure_months",
            "churn_date",
            "churn_reason"
        ]
    ]
    .sort_values("monthly_charge", ascending=False)
    .head(20)
)

display(high_value_churned)


## 20. Customer Risk Segmentation

In [ ]:
# Simple rule-based risk classification for customers who have not churned.
retained_analysis = analysis[analysis["churn_flag"] == 0].copy()

def assign_risk(row):
    if (
        row["login_count"] <= 5
        and row["support_tickets"] >= 3
        and row["monthly_usage"] < 20
    ):
        return "High Risk"
    elif (
        row["login_count"] <= 10
        or row["support_tickets"] >= 3
        or row["monthly_usage"] < 40
    ):
        return "Medium Risk"
    return "Low Risk"

retained_analysis["risk_level"] = retained_analysis.apply(
    assign_risk,
    axis=1
)

risk_summary = (
    retained_analysis.groupby("risk_level")
    .size()
    .reset_index(name="retained_customers")
)

risk_summary["share_of_retained_pct"] = (
    risk_summary["retained_customers"] /
    risk_summary["retained_customers"].sum() * 100
)

risk_order = ["High Risk", "Medium Risk", "Low Risk"]
risk_summary["risk_order"] = pd.Categorical(
    risk_summary["risk_level"],
    categories=risk_order,
    ordered=True
)

risk_summary = risk_summary.sort_values("risk_order").drop(columns="risk_order")

display(risk_summary)


In [ ]:
low_engagement_customers = (
    retained_analysis[
        (retained_analysis["login_count"] <= 5) &
        (retained_analysis["monthly_usage"] < 20)
    ]
    [
        [
            "customer_id",
            "customer_segment",
            "contract_type",
            "monthly_charge",
            "login_count",
            "support_tickets",
            "monthly_usage",
            "last_login_date",
            "tenure_months"
        ]
    ]
    .sort_values("monthly_charge", ascending=False)
    .head(50)
)

display(low_engagement_customers)


## 21. Segment and Contract Analysis

In [ ]:
segment_contract = (
    analysis.groupby(["customer_segment", "contract_type"])
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

segment_contract["churn_rate_pct"] = (
    segment_contract["churned_customers"] /
    segment_contract["customers"] * 100
)

display(
    segment_contract.sort_values("churn_rate_pct", ascending=False)
)


## 22. Executive Summary

In [ ]:
# Generate a concise set of portfolio-ready findings from the calculated results.

highest_segment = segment_analysis.iloc[0]
highest_contract = contract_analysis.iloc[0]
top_reason = churn_reasons.iloc[0] if not churn_reasons.empty else None
top_revenue_segment = revenue_risk_by_segment.iloc[0] if not revenue_risk_by_segment.empty else None

print(f"Total customers: {total_customers:,}")
print(f"Churned customers: {churned_customers:,}")
print(f"Overall churn rate: {churn_rate:.2f}%")
print(f"Overall retention rate: {retention_rate:.2f}%")

print(
    f"\nHighest churn-rate segment: "
    f"{highest_segment['customer_segment']} "
    f"({highest_segment['churn_rate_pct']:.2f}%)"
)

print(
    f"Highest churn-rate contract type: "
    f"{highest_contract['contract_type']} "
    f"({highest_contract['churn_rate_pct']:.2f}%)"
)

if top_reason is not None:
    print(
        f"Most common churn reason: "
        f"{top_reason['churn_reason']} "
        f"({top_reason['churned_customers']:,} customers)"
    )

print(f"Monthly revenue at risk: {monthly_revenue_at_risk:,.2f}")

if top_revenue_segment is not None:
    print(
        f"Segment with highest revenue at risk: "
        f"{top_revenue_segment['customer_segment']} "
        f"({top_revenue_segment['monthly_revenue_at_risk']:,.2f})"
    )


## 23. Business Recommendations

Based on the calculated analysis, the following areas can be considered for retention strategy:

1. Focus retention campaigns on customer segments with the highest churn rates.
2. Review contract types associated with higher churn.
3. Monitor customers with low login activity and low monthly usage.
4. Prioritize high-value churned customers when evaluating win-back opportunities.
5. Investigate the most common churn reasons to identify product or service improvements.
6. Monitor monthly churn and revenue at risk as ongoing retention KPIs.

These recommendations should be interpreted alongside the calculated results rather than treated as causal conclusions.


## Conclusion

This analysis demonstrates a complete Python workflow for Customer Churn & Retention Analysis using the supplied cleaned datasets.

The notebook covers data validation, KPI calculation, segmentation, behavioral analysis, churn reasons, revenue-at-risk analysis, risk segmentation, visualization, and business recommendations.
